# Référence industrielle : OR-Tools vs nos solveurs

Ce notebook compare nos implémentations (Clarke-Wright, ALNS, AG) au solveur
**OR-Tools** de Google, sur la même instance VRPTW.

**Pourquoi c'est important** : un algorithme seul ne prouve rien. Ce qui prouve
quelque chose, c'est l'**écart mesuré** à une référence reconnue. C'est la
différence entre « mon algorithme converge » et « mon algorithme est à 4 % d'un
solveur industriel, en 30 fois moins de temps ».

**Exécution** : Runtime → Run all (environ 2 minutes).


In [ ]:
!pip install -q ortools requests numpy


## 1. Récupérer l'instance réelle

Si le fichier `data/instances/abidjan_communes_12.json` est dans le dépôt, on le
charge directement. Sinon, on le reconstruit depuis OSRM.


In [ ]:
import json, pathlib, subprocess, sys

INSTANCE = pathlib.Path("data/instances/abidjan_communes_12.json")

if not INSTANCE.exists():
    # dans Colab, cloner le dépôt puis générer l'instance
    if not pathlib.Path("abidjan-route-optimization").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/grimael/abidjan-route-optimization.git"], check=True)
    INSTANCE = pathlib.Path("abidjan-route-optimization/data/instances/abidjan_communes_12.json")
    if not INSTANCE.exists():
        subprocess.run([sys.executable, "abidjan-route-optimization/scripts/build_instance.py",
                        "--out", str(INSTANCE)], check=True)

data = json.loads(INSTANCE.read_text(encoding="utf-8"))
print(INSTANCE, "->", len(data["points"]), "points")
print("source :", data.get("source"))


## 2. Construire un VRPTW à partir de l'instance réelle

L'instance réelle ne contient pas de demandes ni de fenêtres de temps : on les
ajoute de façon **reproductible** (graine fixe), en documentant l'hypothèse.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = len(data["points"])
travel = np.array(data["durations"], dtype=float)
demand = np.concatenate([[0], rng.integers(1, 5, n - 1)]).astype(float)
capacity = 12.0
service = np.concatenate([[0], rng.integers(120, 600, n - 1)]).astype(float)
horizon = 8 * 3600
start = rng.uniform(0, horizon - 3 * 3600, n)
start[0] = 0.0
tw = np.column_stack([start, np.minimum(start + 3 * 3600, horizon)])

print(f"{n} clients, capacité {capacity}, horizon {horizon/3600:.0f} h")


## 3. Référence OR-Tools

Modèle standard : dimension temps (avec fenêtres), dimension capacité,
recherche guidée par les coûts avec redémarrages.


In [ ]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

def solve_ortools(travel, demand, capacity, tw, service, horizon, vehicles=4, time_limit=30):
    """Modele VRPTW standard : la dimension temps additionne DEPLACEMENT + SERVICE."""
    n = len(demand)
    manager = pywrapcp.RoutingIndexManager(n, vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)

    def travel_cb(fi, ti):
        return int(travel[manager.IndexToNode(fi), manager.IndexToNode(ti)])

    def time_cb(fi, ti):
        i = manager.IndexToNode(fi)
        j = manager.IndexToNode(ti)
        return int(travel[i, j] + service[i])

    def demand_cb(fi):
        return int(demand[manager.IndexToNode(fi)])

    # cout = temps de deplacement seul
    routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(travel_cb))

    # temps = deplacement + service (c'est la que se joue le respect des fenetres)
    # l'attente maximale par client doit couvrir tout l'horizon, sinon OR-Tools
    # interdit les tournees que notre ALNS accepte (attentes de plusieurs heures)
    routing.AddDimension(routing.RegisterTransitCallback(time_cb), int(horizon), int(horizon), False, "Time")
    dim = routing.GetDimensionOrDie("Time")
    for node in range(1, n):
        idx = manager.NodeToIndex(node)
        dim.CumulVar(idx).SetRange(int(tw[node][0]), int(tw[node][1]))

    routing.AddDimensionWithVehicleCapacity(
        routing.RegisterUnaryTransitCallback(demand_cb), 0, [int(capacity)] * vehicles, True, "Capacity"
    )

    for v in range(vehicles):
        dim.CumulVar(routing.Start(v)).SetRange(0, int(horizon))
        routing.AddVariableMinimizedByFinalizer(dim.CumulVar(routing.Start(v)))
        routing.AddVariableMinimizedByFinalizer(dim.CumulVar(routing.End(v)))

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.time_limit.FromSeconds(time_limit)

    solution = routing.SolveWithParameters(params)
    if not solution:
        return None, None
    routes = []
    for v in range(vehicles):
        route, index = [], routing.Start(v)
        while not routing.IsEnd(index):
            route.append(manager.IndexToNode(index))
            index = solution.Value(routing.NextVar(index))
        routes.append(route[1:])
    return [r for r in routes if r], int(solution.ObjectiveValue())

import time
t0 = time.perf_counter()
routes_ortools, cost_ortools = solve_ortools(travel, demand, capacity, tw, service, horizon)
dt = time.perf_counter() - t0
print(f"OR-Tools : cout {cost_ortools} s de deplacement, {len(routes_ortools)} vehicules, {dt:.1f} s de calcul")


## 4. Nos solveurs sur la même instance


In [ ]:
import sys, time
sys.path.insert(0, "abidjan-route-optimization/src")
sys.path.insert(0, "src")

from vrp.instance import Instance
from vrp.solvers.vrptw import alns_tw, route_schedule, solomon_insertion, summary

inst = Instance(
    name="abidjan_vrptw", lat=np.array([p["lat"] for p in data["points"]]),
    lon=np.array([p["lon"] for p in data["points"]]), demand=demand,
    capacity=capacity, tw=tw, service=service, matrix=travel,
    cost_unit="s", source=data.get("source", ""),
    meta={"points": data["points"]},
)

t0 = time.perf_counter()
base = solomon_insertion(inst)
t_insert = time.perf_counter() - t0
t0 = time.perf_counter()
best = alns_tw(inst, start=base, iterations=300, seed=1)
t_alns = time.perf_counter() - t0

print("insertion :", summary(inst, base), f"{t_insert:.1f} s")
print("ALNS      :", summary(inst, best), f"{t_alns:.1f} s")

if routes_ortools:
    violations = [r for r in routes_ortools if not route_schedule(inst, r)[0]]
    ok = len(routes_ortools) - len(violations)
    print(f"Controle croise : {ok}/{len(routes_ortools)} tournees OR-Tools faisables selon notre simulateur")
else:
    print("Controle croise impossible : OR-Tools n'a rendu aucune solution.")


## 5. Comparaison


In [ ]:
c_base = sum(inst.tour_cost(r) for r in base)
c_alns = sum(inst.tour_cost(r) for r in best)

if cost_ortools is None:
    print("OR-Tools n'a trouve aucune solution faisable dans la limite de temps.")
    print("Relancer avec time_limit=90 dans la cellule precedente.")
else:
    print(f"{'Solveur':<24}{'Cout (s)':>10}{'Ecart OR-Tools':>16}{'Temps':>9}")
    print("-" * 60)
    for name, c, t in [("Insertion Solomon", c_base, t_insert),
                       ("ALNS maison", c_alns, t_alns)]:
        gap = (c - cost_ortools) / cost_ortools * 100
        print(f"{name:<24}{c:>10.0f}{gap:>15.1f}%{t:>8.1f}s")
    print(f"{'OR-Tools (reference)':<24}{cost_ortools:>10.0f}{0.0:>15.1f}%{dt:>8.1f}s")
    print()
    print("Ecart negatif = notre solution est meilleure (plus courte) que celle d'OR-Tools.")
    print("Ecart positif = OR-Tools fait mieux, ce qui est le cas attendu sur une instance facile.")


## 6. Lecture honnête des résultats

- Si l'ALNS est **à moins de 10 %** d'OR-Tools, c'est un excellent résultat pour
  une implémentation de moins de 200 lignes, sans dépendance.
- OR-Tools gagne généralement sur la qualité **et** sur le temps : c'est normal,
  il est développé depuis 2014 par une équipe dédiée.
- L'intérêt scientifique n'est pas de battre OR-Tools, mais de **mesurer l'écart**
  et de comprendre **pourquoi** : qualité de la recherche locale, gestion des
  fenêtres de temps, critère d'acceptation.

**À reporter dans le README** : la table ci-dessus, avec la date et la version
d'OR-Tools, pour que le résultat soit reproductible.
